# Correlation and credit portfolio models

**Prerequisites:** work through `01_foundations/core_types_and_money.ipynb` and `01_foundations/math_toolkit.ipynb`. This notebook uses `finstack_quant.valuations.correlation` and `finstack_quant.core.math.special_functions` for $\Phi^{-1}(\mathrm{PD})$ and $t_\nu^{-1}(\mathrm{PD})$ where needed.

**Scope:** one-factor copulas, two-name Bernoulli correlation, recovery specs, simple factor structures, and matrix checks — the usual building blocks for portfolio credit and structured credit intuition.

## Why correlation matters in credit portfolios

For a **single name**, risk is mostly about default probability (PD) and loss given default (LGD). For a **portfolio**, **defaults co-move**: a bad macro draw raises many conditional PDs at once. Models capture that with:

- A **systematic factor** (or factors) representing the market state.
- **Asset correlation** linking each obligor to that factor.
- A **copula** mapping factor realizations to joint default behavior; **Gaussian** copulas have **no tail dependence**, while **Student-$t$** copulas put more mass in joint extreme events — a key distinction for tranches and stress testing.

This notebook walks through the Python API and ends with a small comparison of **conditional default probabilities** under Gaussian vs Student-$t$ (5 df) copulas across factor values $Z$.

### API: imports

We pull correlation types from `finstack_quant.valuations.correlation` and inverse CDFs from `finstack_quant.core.math.special_functions`: $\Phi^{-1}(\mathrm{PD})$ for Gaussian copulas and $t_\nu^{-1}(\mathrm{PD})$ for Student-$t$ copulas.

In [1]:
from finstack_quant.core.math.special_functions import (
    standard_normal_inv_cdf,
    student_t_cdf,
    student_t_inv_cdf,
)
from finstack_quant.valuations.correlation import (
    CopulaSpec,
    CorrelatedBernoulli,
    LatentFactorSpec,
    LatentMultiFactor,
    RecoverySpec,
    LatentSingleFactor,
    LatentTwoFactor,
    cholesky_decompose,
    correlation_bounds,
    joint_probabilities,
    validate_correlation_matrix,
    nearest_correlation,
)

print(
    "Imports loaded: finstack_quant.valuations.correlation (incl. LatentMultiFactor) + inv/normal and Student-t CDFs"
)

Imports loaded: finstack_quant.valuations.correlation (incl. LatentMultiFactor) + inv/normal and Student-t CDFs


### `CopulaSpec` and `Copula`

Build a concrete copula with `.build()`. For `conditional_default_prob(threshold, factor_realization, correlation)`:

- **threshold** — default barrier: $c = \Phi^{-1}(\mathrm{PD})$ for the **Gaussian** copula, and $c = t_\nu^{-1}(\mathrm{PD})$ for the **Student-$t$** copula (same `conditional_default_prob` signature).
- **factor_realization** — e.g. `[Z]` for the systematic factor draw.
- **correlation** — asset correlation $\rho$ (not the PD).

`tail_dependence(rho)` summarizes how much joint mass sits in the lower tail; the one-factor **Gaussian** copula reports **zero** tail dependence.

In [2]:
import math

gauss_spec = CopulaSpec.gaussian()
gauss = gauss_spec.build()
print(f"Model: {gauss.model_name}")
print(f"Factors: {gauss.num_factors}")

pd_demo = 0.03
rho_demo = 0.20
df_demo = 5.0
c_gauss = standard_normal_inv_cdf(pd_demo)
c_t = student_t_inv_cdf(pd_demo, df_demo)
rt_pd = student_t_cdf(c_t, df_demo)
assert math.isclose(rt_pd, pd_demo, rel_tol=0.0, abs_tol=1e-9), (
    f"Student-t PD roundtrip failed: t_cdf(t_inv(pd), df)={rt_pd} vs pd={pd_demo}"
)
print(
    f"Cond. default prob Gauss (Z=-1.5, PD={pd_demo}, rho={rho_demo}): "
    f"{gauss.conditional_default_prob(c_gauss, [-1.5], rho_demo):.6f}"
)

t_spec = CopulaSpec.student_t(df_demo)
t_copula = t_spec.build()
print(
    f"Cond. default prob Student-t({df_demo:.0f}) (Z=-1.5, W=1, same PD, rho={rho_demo}): "
    f"{t_copula.conditional_default_prob(c_t, [-1.5, 1.0], rho_demo):.6f}"
)
print(f"Tail dependence (rho=0.5): {gauss.tail_dependence(0.5):.6f}")
print(f"T-copula tail dep (rho=0.5): {t_copula.tail_dependence(0.5):.6f}")

rfl = CopulaSpec.random_factor_loading(0.2).build()
print(f"RFL model: {rfl.model_name}")

Model: One-Factor Gaussian Copula
Factors: 1
Cond. default prob Gauss (Z=-1.5, PD=0.03, rho=0.2): 0.088061
Cond. default prob Student-t(5) (Z=-1.5, W=1, same PD, rho=0.2): 0.025149
Tail dependence (rho=0.5): 0.000000
T-copula tail dep (rho=0.5): 0.207031
RFL model: Random Factor Loading Copula


### `CorrelatedBernoulli`

A minimal **two-obligor** model: marginals $P(X_1{=}1)=p_1$, $P(X_2{=}1)=p_2$, and a **correlation** between the Bernoulli indicators. Use it to inspect **joint probabilities** and **conditional** quantities such as $P(X_2{=}1 \mid X_1{=}1)$.

In [3]:
cb = CorrelatedBernoulli(0.03, 0.05, 0.20)
print(f"Joint probs (p11,p10,p01,p00): {cb.joint_probabilities()}")
print(f"P(both default): {cb.joint_p11:.6f}")
print(f"P(A defaults | B defaults): {cb.conditional_p2_given_x1():.6f}")

Joint probs (p11,p10,p01,p00): (0.00893572457800852, 0.021064275421991477, 0.04106427542199148, 0.9289357245780085)
P(both default): 0.008936
P(A defaults | B defaults): 0.297857


### Fréchet–Hoeffding: `correlation_bounds` and `joint_probabilities`

For two Bernoullis with given marginals, only some correlations are feasible. **`correlation_bounds`** returns admissible $[\rho_{\min}, \rho_{\max}]$. When PDs match, the upper bound can reach 1. **`joint_probabilities`** returns the four-cell joint distribution for a chosen $\rho$.

In [4]:
bounds = correlation_bounds(0.03, 0.05)
print(f"Correlation bounds for PD1=3%, PD2=5%: {bounds}")

bounds2 = correlation_bounds(0.05, 0.05)
print(f"Same PDs (5%, 5%): {bounds2}")

jp = joint_probabilities(0.03, 0.05, 0.10)
print(f"Joint default prob (PD1=3%, PD2=5%, rho=10%): {jp}")

Correlation bounds for PD1=3%, PD2=5%: (-0.040345765480241574, 0.7665695441245898)
Same PDs (5%, 5%): (-0.05263157894736843, 1.0)
Joint default prob (PD1=3%, PD2=5%, rho=10%): (0.00521786228900426, 0.024782137710995737, 0.04478213771099574, 0.9252178622890043)


### `RecoverySpec`: constant vs market-correlated recovery

**LGD** is $1 - \mathrm{recovery}$ (for the constant case). Stochastic recovery links recovery to the same systematic shock as credit quality, which changes tail loss — relevant for CDO and index tranche work.

In [5]:
rec_const = RecoverySpec.constant(0.40)
rec_model = rec_const.build()
print(f"Expected recovery: {rec_model.expected_recovery:.2f}")
print(f"LGD: {rec_model.lgd:.2f}")
print(f"Is stochastic: {rec_model.is_stochastic}")
print(f"Conditional recovery (Z=-2): {rec_model.conditional_recovery(-2.0):.4f}")

rec_stoch = RecoverySpec.market_correlated(0.40, 0.10, 0.50)
rec_stoch_model = rec_stoch.build()
print(
    f"Stochastic recovery: expected={rec_stoch_model.expected_recovery:.4f}"
)
print(f"  Conditional (Z=-2): {rec_stoch_model.conditional_recovery(-2.0):.4f}")
print(f"  Conditional (Z=+2): {rec_stoch_model.conditional_recovery(2.0):.4f}")

Expected recovery: 0.40
LGD: 0.60
Is stochastic: False
Conditional recovery (Z=-2): 0.4000
Stochastic recovery: expected=0.4010
  Conditional (Z=-2): 0.3053
  Conditional (Z=+2): 0.5028


### Factor specs and multi-name structures

`LatentFactorSpec` defers construction like copula specs; **`LatentSingleFactor`** and **`LatentTwoFactor`** are concrete. **`LatentTwoFactor.clo_standard`** and **`rmbs_standard`** ship common structured-credit calibrations.

In [6]:
sf = LatentFactorSpec.single_factor(0.15, 0.5)
sf_model = sf.build()
print(f"SingleFactor (via LatentFactorSpec): {sf_model}")

sfm = LatentSingleFactor(0.15, 0.5)
print(f"Direct LatentSingleFactor: {sfm}")

tf = LatentTwoFactor.clo_standard()
print(f"CLO standard: {tf}")

tf_rmbs = LatentTwoFactor.rmbs_standard()
print(f"RMBS standard: {tf_rmbs}")

SingleFactor (via LatentFactorSpec): LatentFactorKind('Single Factor Model', n=1)
Direct LatentSingleFactor: LatentSingleFactor(vol=0.1500, mr=0.5000)
CLO standard: LatentTwoFactor(prepay=0.1500, credit=0.3000, corr=-0.2000)
RMBS standard: LatentTwoFactor(prepay=0.2000, credit=0.2500, corr=-0.3000)


### Matrix validation and Cholesky

Portfolio simulation often needs a **valid correlation matrix** and a **Cholesky** factor for correlated normals. Here the API expects a **row-major flat** vector of length $n^2$.

In [7]:
validate_correlation_matrix([1.0, 0.3, 0.3, 1.0], 2)
print("Valid 2x2 (rho=0.3): validation passed")

L = cholesky_decompose([1.0, 0.5, 0.5, 1.0], 2)
print(f"Cholesky L (flat): {L}")

nc = nearest_correlation([1.0, 0.99, 0.99, 1.0], 2)
print("nearest_correlation (already PSD):", nc[:4])

Valid 2x2 (rho=0.3): validation passed
Cholesky L (flat): [0.8660254037844386, 0.5, 0.0, 1.0]
nearest_correlation (already PSD): [1.0, 0.99, 0.99, 1.0]


### Mini-example: Gaussian vs Student-$t$(5) conditional PDs

Fix $\mathrm{PD}=3\%$ and $\rho=0.3$. For each systematic draw, compare $P(\mathrm{default}\mid M)$ under the two copulas: **Gaussian** uses factor $Z\sim\mathcal{N}(0,1)$ with threshold $\Phi^{-1}(\mathrm{PD})$; **Student-$t$** uses the same factor input $M$ passed to `conditional_default_prob` with threshold $t_\nu^{-1}(\mathrm{PD})$ (here $\nu=5$). You should see the largest **gaps in the tails** (very negative factor draws): the $t$-copula puts more weight on joint bad outcomes than the Gaussian copula at the same linear correlation.

In [8]:
import math

z_values = [-3.0, -2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0, 3.0]
pd = 0.03
rho = 0.3
df_table = 5.0
c_gauss = standard_normal_inv_cdf(pd)
c_t = student_t_inv_cdf(pd, df_table)
assert math.isclose(student_t_cdf(c_t, df_table), pd, rel_tol=0.0, abs_tol=1e-9), (
    "Student-t PD roundtrip: t_cdf(t_inv(pd), df) should match pd"
)

print(
    f"PD={pd}, rho={rho}: Gaussian threshold Phi^-1(PD)={c_gauss:.6f}; "
    f"Student-t({df_table:.0f}) threshold t^-1(PD)={c_t:.6f}"
)
print(f"Gaussian tail_dependence(rho): {gauss.tail_dependence(rho):.6f}")
print(f"Student-t(5) tail_dependence(rho): {t_copula.tail_dependence(rho):.6f}")
print("Student-t column conditions on the shared mixing factor W=1.")
print()

print(f"{'Z':>6} {'Gaussian':>12} {'Student-t(5)':>14}")
print("-" * 34)
for z in z_values:
    g_prob = gauss.conditional_default_prob(c_gauss, [z], rho)
    t_prob = t_copula.conditional_default_prob(c_t, [z, 1.0], rho)
    print(f"{z:>6.1f} {g_prob:>12.6f} {t_prob:>14.6f}")

PD=0.03, rho=0.3: Gaussian threshold Phi^-1(PD)=-1.880794; Student-t(5) threshold t^-1(PD)=-2.421585
Gaussian tail_dependence(rho): 0.000000
Student-t(5) tail_dependence(rho): 0.122387
Student-t column conditions on the shared mixing factor W=1.

     Z     Gaussian   Student-t(5)
----------------------------------
  -3.0     0.388199       0.176086
  -2.0     0.173950       0.056479
  -1.5     0.102757       0.027915
  -1.0     0.055544       0.012555
  -0.5     0.027388       0.005129
   0.0     0.012289       0.001900
   0.5     0.005008       0.000637
   1.0     0.001850       0.000193
   1.5     0.000619       0.000053
   2.0     0.000187       0.000013
   3.0     0.000013       0.000001


## Takeaways

- **Copulas** share `conditional_default_prob(threshold, [factor], rho)`; **threshold** is $\Phi^{-1}(\mathrm{PD})$ for **Gaussian** and $t_\nu^{-1}(\mathrm{PD})$ for **Student-$t$**, not the factor draw.
- **Gaussian** vs **Student-$t$** differ most in the **tails**; tail dependence is **zero** for the Gaussian copula in this implementation but **positive** for the $t$-copula — a standard reason to use $t$ copulas in credit-tranche calibration.
- **Two-name** tools (`CorrelatedBernoulli`, `correlation_bounds`, `joint_probabilities`) encode **feasible** correlation and **joint** default mass for simple portfolios.
- **Recovery** can be **constant** or **factor-linked**; stochastic recovery changes how losses concentrate in stress scenarios.
- **Factor models** (single, CLO/RMBS presets, multi-factor) support simulation and structured-credit-style correlation structure beyond a single market factor.

Run all cells top to bottom after installing the `finstack_quant` Python package built from this repo.